# Part 2: How Real VLAs Represent Actions

## Notebook 4 — Diffusion Policy

Diffusion Policy (Chi et al., RSS 2023) generates **continuous action chunks** by iteratively denoising from Gaussian noise. Actions are never discretized — they emerge from a learned denoising process.

We load the leRobot DiffusionPolicy and inspect its noise-based generation.


### 1. Load Diffusion Policy configuration

Diffusion Policy uses a U-Net conditioned on observations and a diffusion timestep to predict either the noise or the clean action.


In [1]:
from lerobot.policies.diffusion.configuration_diffusion import DiffusionConfig

cfg = DiffusionConfig()
print(f"Policy type: Diffusion Policy")
print(f"Action steps:          {cfg.n_action_steps}")  # 32
print(f"Noise scheduler:       {cfg.noise_scheduler_type}")  # DDPM
print(f"Train timesteps:       {cfg.num_train_timesteps}")
print(f"Diffusion embed dim:   {cfg.diffusion_step_embed_dim}")  # 128
print(f"Prediction type:       {cfg.prediction_type}")  # epsilon
print(f"Inference steps:       {cfg.num_inference_steps}")


Policy type: Diffusion Policy
Action steps:          32
Noise scheduler:       DDPM
Train timesteps:       100
Diffusion embed dim:   128
Prediction type:       epsilon
Inference steps:       None


### 2. How diffusion generates actions

Training: add noise to real actions → train model to predict noise.
Inference: start from pure noise → iteratively denoise → action chunk.


In [2]:
import torch

# Diffusion Policy action generation = iterative denoising
print("Training:")
print("  1. Take real action chunk a_0 (32 steps × 7 dims)")
print("  2. Sample timestep t ~ Uniform(0, T)")
print("  3. Add noise: a_t = √(ᾱ_t) * a_0 + √(1-ᾱ_t) * ε")
print("  4. Train model to predict ε from a_t")

print("Inference (generate_actions):")
print("  1. Sample a_T ~ N(0, I)  ← pure noise")
print("  2. For t = T to 1:")
print("       model predicts ε = f(a_t, observation)")
print("       denoise: a_{t-1} = (a_t - √(1-ᾱ_t)ε) / √(ᾱ_t)")
print("  3. a_0 = clean action chunk (32 × 7 continuous values)")


Training:
  1. Take real action chunk a_0 (32 steps × 7 dims)
  2. Sample timestep t ~ Uniform(0, T)
  3. Add noise: a_t = √(ᾱ_t) * a_0 + √(1-ᾱ_t) * ε
  4. Train model to predict ε from a_t
Inference (generate_actions):
  1. Sample a_T ~ N(0, I)  ← pure noise
  2. For t = T to 1:
       model predicts ε = f(a_t, observation)
       denoise: a_{t-1} = (a_t - √(1-ᾱ_t)ε) / √(ᾱ_t)
  3. a_0 = clean action chunk (32 × 7 continuous values)


### 3. Action shape: continuous, no tokens

Like ACT, Diffusion Policy outputs raw continuous vectors. The difference is the generation process: sample-and-decode (CVAE) vs iterative denoising (diffusion).


In [4]:
# Action shape from Diffusion Policy
batch_size = 1
horizon = cfg.n_action_steps  # 32
action_dim = 7

# What generate_actions() returns
actions = torch.randn(batch_size, horizon, action_dim)
print(f"Diffusion Policy action shape: {actions.shape}")  # [1, 32, 7]
print(f"Total values: {actions.numel()}")  # 224

# Compare with other policies
print("\nAction shapes across policies:")
print(f"  ACT:              (1, 100, 7) = 700 values")  # chunk_size=100
print(f"  Diffusion Policy: (1, 32, 7)  = 224 values")  # n_action_steps=32
print(f"  pi0:              (1, 50, 7)  = 350 values")  # chunk_size=50
print(f"  pi0-FAST:         ~ 30-60 FAST tokens  (after DCT+BPE)")


Diffusion Policy action shape: torch.Size([1, 32, 7])
Total values: 224

Action shapes across policies:
  ACT:              (1, 100, 7) = 700 values
  Diffusion Policy: (1, 32, 7)  = 224 values
  pi0:              (1, 50, 7)  = 350 values
  pi0-FAST:         ~ 30-60 FAST tokens  (after DCT+BPE)


### 4. Diffusion vs ACT: why different approaches?

ACT (CVAE): single forward pass, fast inference (~30 Hz).
Diffusion: multiple denoising steps, slower inference (~10-30 Hz).

But diffusion handles multi-modal distributions more naturally — it can represent arbitrarily complex action distributions without the KL divergence bottleneck of a VAE.


In [3]:
# Trade-off summary
print("ACT (CVAE):")
print("  + Fast single-pass inference")
print("  + Simple training")
print("  - KL divergence bottleneck limits distribution complexity")

print("Diffusion Policy:")
print("  + Unconstrained action distributions")
print("  + Very smooth trajectories")
print("  - Slower inference (T denoising steps)")
print("  - More hyperparameters (noise schedule, steps)")


ACT (CVAE):
  + Fast single-pass inference
  + Simple training
  - KL divergence bottleneck limits distribution complexity
Diffusion Policy:
  + Unconstrained action distributions
  + Very smooth trajectories
  - Slower inference (T denoising steps)
  - More hyperparameters (noise schedule, steps)


### Key Takeaway

Diffusion Policy generates continuous actions through iterative denoising. No tokens, no bins, no vocabulary. The action emerges from the denoising process. This approach produces exceptionally smooth trajectories but trades off inference speed.
